In [1]:
# Importing the necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re
import joblib
import string


In [2]:
# Loading the datasets: 'Fake.csv' contains fake news and 'True.csv' contains real news
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

In [3]:
# Displaying the first five rows of the 'fake' dataset to get a quick overview
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [4]:
# Displaying the first five rows of the 'true' dataset to get a quick overview
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [5]:
# Adding a new column 'class' to label the data: 0 for fake news, 1 for real news
fake['class']=0
true['class']=1

In [6]:
# Combining both datasets into one DataFrame along the rows (axis=0)
data = pd.concat([fake, true], axis=0)

In [7]:
# Displaying a random sample of 10 rows from the combined dataset to inspect mixed entries
data.sample(10)

,title,text,subject,date,class
10991,Connecticut's governor says no return to pre-r...,NEW YORK (Reuters) - Connecticut Governor Dann...,politicsNews,"February 3, 2016",1
15187,"OBAMA’S PLAN EXPOSED: 115,000 IMMIGRANTS In Te...",The fundamental transformation of America: Wat...,politics,"Sep 21, 2015",0
14671,China foreign minister to visit Myanmar amid R...,BEIJING (Reuters) - Chinese Foreign Minister W...,worldnews,"November 16, 2017",1
17954,Russia threatens to brand U.S.-sponsored Radio...,MOSCOW (Reuters) - U.S. government-sponsored R...,worldnews,"October 9, 2017",1
4740,No. 2 House Republican says healthcare bill de...,WASHINGTON (Reuters) - The No. 2 Republican in...,politicsNews,"March 23, 2017",1
6909,North Carolina governor concedes election to D...,"WINSTON-SALEM, N.C. (Reuters) - North Carolina...",politicsNews,"December 5, 2016",1
20998,HERE WE GO AGAIN…GRAMMY AWARDS UNDER FIRE For ...,All of a sudden #BlackDeathsMatter Could the G...,left-news,"Feb 15, 2016",0
21882,ONLY IN DETROIT: SQUATTING ON THE SQUATTER TAK...,You won t want to miss this,left-news,"Apr 9, 2015",0
896,Trump taps Fed centrist Powell to lead U.S. ce...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"November 2, 2017",1
45,White House Panics Knowing Flynn Is Going To ...,"While Donald Trump has been taking vacations, ...",News,"December 1, 2017",0


In [8]:
# Dropping unnecessary columns that are not needed for fake news detection
data = data.drop(["subject", "title", "date"], axis = 1)

In [9]:
# Resetting the index after dropping columns to keep the DataFrame tidy
data.reset_index(inplace=True)

In [10]:
# Dropping the old index column that was added during reset
data.drop(['index'],axis =1 , inplace=True)

In [11]:
# Displaying a random sample of 5 rows from the cleaned dataset
data.sample(5)

,text,class
22701,21st Century Wire says A new front has just op...,0
19238,The Washington Post reported that higher ups i...,0
40170,LUXEMBOURG/BRUSSELS (Reuters) - Most European ...,1
32137,SAN DIEGO (Reuters) - A U.S. judge on Friday t...,1
23055,Patrick Henningsen 21st Century WireWatching t...,0


In [12]:
# Function to clean the input text by removing unwanted characters and formatting
def clean_text(text):
    text = text.lower()  # Convert all text to lowercase for uniformity
    text = re.sub(r'\[.*?\]', '', text)  # Remove text within square brackets
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'<.*?>+', '', text)  # Remove HTML tags
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)  # Remove punctuation
    text = re.sub(r'\w*\d\w*', '', text)  # Remove words containing numbers
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace and trim the text
    return text  # Return the cleaned text


In [13]:
# Applying the clean_text function to preprocess all text data
data['text'] = data['text'].apply(clean_text)

In [14]:
# Separating features and labels
x = data['text']
y = data['class']

# Splitting the dataset into training and testing sets (75% train, 25% test)
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.25, random_state=4)

In [17]:
# Initializing the TF-IDF Vectorizer
vectorizer = TfidfVectorizer()
xv_train = vectorizer.fit_transform(xtrain)
xv_test = vectorizer.transform(xtest)

In [16]:
vectorizer = TfidfVectorizer()
xv_train = vectorizer.fit_transform(xtrain)
xv_test = vectorizer.transform(xtest)

In [18]:
# Initializing the Logistic Regression model and training it 
lr = LogisticRegression()
lr.fit(xv_train, ytrain)

LogisticRegression()

In [19]:
# Making predictions on the test data using the trained Logistic Regression model
prediction = lr.predict(xv_test)

# Calculating and returning the accuracy score of the model on the test set
lr.score(xv_test, ytest)

0.985924276169265

In [20]:
print(classification_report(ytest, prediction))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5952
           1       0.98      0.99      0.99      5273

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [21]:
joblib.dump(vectorizer, "vectorizer.jb")
joblib.dump(lr, "lr_model.jb")

['lr_model.jb']